# 02 — Preprocessing

**Project:** MentalBERT-CSSR  
**Goal of this notebook:** Apply *conservative* cleaning to CSSR-S Reddit posts and write `DATA/processed/cssrs_processed.csv`.

---

## Clinical-signal preservation policy

| Action | Decision | Rationale |
|--------|----------|-----------|
| Negations (`not`, `n't`, `never`, `no`) | **Keep** | Polarity flips suicidal intent statements |
| Emotion / affect words | **Keep** | Core severity signal |
| Punctuation (`!`, `?`, `...`) | **Keep** | Intensity / uncertainty cues |
| Emoji | **Keep** | Affective signal in social media |
| Stopword removal / stemming / lemmatisation | **Do not** | Destroys meaning; MentalBERT expects natural text |
| Lowercasing | **Do not** (here) | Uncased tokenizer lowercases at encode-time; store text faithfully |
| `severity` labels | **Never modify** | Human ground truth |
| LLM label columns | **Retain in CSV, ignore in training** | Benchmarking only |

## Cleaning operations (only what is necessary)

1. HTML entity unescape (meaning-preserving)
2. Unicode normalisation (**NFC**)
3. Remove invalid / control / zero-width characters
4. Collapse repeated whitespace; strip ends
5. Drop null / blank `content` or `severity` rows
6. Drop rows whose severity is outside `{0…6}` (no remapping)
7. Drop exact duplicate rows and duplicate `content` strings

> **Stop after this notebook** until Notebook 3 (Tokenization) is approved.

## 1. Environment, configuration, and reproducibility

In [ ]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

NOTEBOOK_DIR = Path.cwd().resolve()
CANDIDATES = [NOTEBOOK_DIR, NOTEBOOK_DIR.parent]
PROJECT_ROOT = None
for candidate in CANDIDATES:
    if (candidate / "configs" / "default.yaml").exists() and (candidate / "utils").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate project root (expected configs/default.yaml and utils/)."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils import ensure_directories, get_logger, load_config, set_seed
from utils.eda import class_distribution, plot_severity_distribution
from utils.io import load_raw_dataset, save_processed_dataset, training_columns
from utils.preprocessing import (
    before_after_examples,
    clean_text,
    preprocess_dataframe,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", 160)

cfg = load_config()
set_seed(cfg.SEED)
ensure_directories(cfg)
logger = get_logger("notebook.02_preprocessing")

TEXT_COL = cfg.data.text_column
LABEL_COL = cfg.data.label_column
PLOTS_DIR = Path(cfg.paths.plots_path)
METRICS_DIR = Path(cfg.paths.metrics_path)
PROCESSED_PATH = Path(cfg.paths.processed_data)
PP = cfg.preprocessing

logger.info("Project root     : %s", PROJECT_ROOT)
logger.info("Raw data         : %s", cfg.paths.raw_data)
logger.info("Processed output : %s", PROCESSED_PATH)
logger.info("Unicode form     : %s", PP.unicode_form)

## 2. Load raw data

Raw CSV remains the immutable source of truth. All cleaning writes to `DATA/processed/`.

In [ ]:
df_raw = load_raw_dataset(
    path=cfg.paths.raw_data,
    text_column=TEXT_COL,
    label_column=LABEL_COL,
)

print(f"Raw shape: {df_raw.shape}")
print(f"Columns : {list(df_raw.columns)}")
display(df_raw[[TEXT_COL, LABEL_COL]].head(3))

# Snapshot label distribution BEFORE cleaning (should match EDA)
dist_before = class_distribution(df_raw, label_column=LABEL_COL)
print("Severity distribution (raw):")
display(dist_before)

## 3. Unit checks on `clean_text`

Verify the cleaner keeps negations / punctuation / emotion tokens while fixing whitespace and control noise.

In [ ]:
unit_cases = [
    # negations + intensity punctuation must survive
    "I do NOT want to die!!! but I can't keep going...",
    # repeated whitespace / tabs / newlines
    "help\t\tme\n\nplease   now",
    # HTML entities
    "I feel &lt;empty&gt; &amp; hopeless",
    # zero-width chars + control noise
    "still here\u200b\u200b tomorrow\x00",
    # emotion words + emoji retained
    "I am so sad and scared 😢 please",
]

unit_rows = []
for raw in unit_cases:
    unit_rows.append({"raw": raw, "cleaned": clean_text(raw, unicode_form=PP.unicode_form, unescape_html=PP.unescape_html)})

unit_df = pd.DataFrame(unit_rows)
display(unit_df)

# Hard assertions — fail the notebook early if the policy regresses
assert "NOT" in unit_df.loc[0, "cleaned"]
assert "can't" in unit_df.loc[0, "cleaned"]
assert "!!!" in unit_df.loc[0, "cleaned"]
assert "..." in unit_df.loc[0, "cleaned"]
assert "  " not in unit_df.loc[1, "cleaned"]
assert "<empty>" in unit_df.loc[2, "cleaned"]
assert "&" in unit_df.loc[2, "cleaned"]
assert "\x00" not in unit_df.loc[3, "cleaned"]
assert "sad" in unit_df.loc[4, "cleaned"] and "scared" in unit_df.loc[4, "cleaned"]
print("Unit checks passed.")

## 4. Run full preprocessing pipeline

Implementation lives in `utils/preprocessing.py` so Notebooks 3–5 can reload the same logic without duplication.

In [ ]:
output_columns = list(PP.output_columns)
# Only request columns that exist in the raw file
output_columns = [c for c in output_columns if c in df_raw.columns]

df_processed, report = preprocess_dataframe(
    df_raw,
    text_column=TEXT_COL,
    label_column=LABEL_COL,
    valid_labels=list(range(cfg.NUM_LABELS)),
    unicode_form=PP.unicode_form,
    unescape_html=bool(PP.unescape_html),
    drop_exact_duplicates=bool(PP.drop_exact_duplicates),
    drop_duplicate_text=bool(PP.drop_duplicate_text),
    keep_columns=output_columns,
)

report_dict = report.to_dict()
print("PREPROCESS REPORT")
display(pd.DataFrame([{k: v for k, v in report_dict.items() if not isinstance(v, list)}]))
print("Operations:", report_dict["operations"])
print(f"Rows in → out: {report.n_input_rows} → {report.n_output_rows} "
      f"(dropped {report.n_input_rows - report.n_output_rows})")

## 5. Integrity checks

Confirm labels were not remapped and training-critical fields are clean.

In [ ]:
# Labels must remain in {0..6} with identical semantics
unique_labels = sorted(df_processed[LABEL_COL].unique().tolist())
print("Processed severity labels:", unique_labels)
assert df_processed[LABEL_COL].between(0, cfg.NUM_LABELS - 1).all()
assert set(unique_labels).issubset(set(range(cfg.NUM_LABELS)))

# No nulls / blanks in training fields
assert df_processed[TEXT_COL].isna().sum() == 0
assert df_processed[LABEL_COL].isna().sum() == 0
assert (df_processed[TEXT_COL].str.strip().str.len() > 0).all()

# No duplicate content after pipeline
assert df_processed[TEXT_COL].duplicated().sum() == 0

# Compare class counts: cleaning should not systematically erase a class
dist_after = class_distribution(df_processed, label_column=LABEL_COL)
compare = dist_before.merge(
    dist_after,
    on="severity",
    how="outer",
    suffixes=("_raw", "_processed"),
).fillna(0)
compare["count_raw"] = compare["count_raw"].astype(int)
compare["count_processed"] = compare["count_processed"].astype(int)
compare["delta"] = compare["count_processed"] - compare["count_raw"]
display(compare[["severity", "count_raw", "count_processed", "delta"]])

print("Integrity checks passed.")

## 6. Before / after text examples

Manual QA: confirm we did not wipe negations or affect language on real posts.

In [ ]:
if bool(PP.save_before_after_examples):
    examples = before_after_examples(
        df_raw,
        text_column=TEXT_COL,
        n=int(PP.n_before_after_examples),
        seed=cfg.SEED,
    )
    display(examples)
    examples_path = METRICS_DIR / "preprocessing_before_after_examples.csv"
    examples.to_csv(examples_path, index=False, encoding="utf-8")
    logger.info("Wrote before/after examples → %s", examples_path)
else:
    examples = pd.DataFrame()
    print("Before/after example export disabled in config.")

## 7. Processed distribution plot + training view preview

In [ ]:
plot_severity_distribution(
    dist_after,
    output_path=PLOTS_DIR / "preprocessing_severity_distribution.png",
    dpi=int(cfg.eda.figure_dpi),
    title="Severity Distribution After Preprocessing",
)

train_view = training_columns(
    df_processed,
    text_column=TEXT_COL,
    label_column=LABEL_COL,
    ignore_columns=list(cfg.data.ignore_columns),
)
print("Training view columns (LLM labels excluded):", list(train_view.columns))
print("Training view shape:", train_view.shape)
display(train_view.head(5))

# Length sanity after cleaning
word_counts = train_view[TEXT_COL].str.split().str.len()
print(
    f"Word count after clean | min={int(word_counts.min())} "
    f"mean={word_counts.mean():.2f} max={int(word_counts.max())} "
    f"p95={word_counts.quantile(0.95):.2f} p99={word_counts.quantile(0.99):.2f}"
)

## 8. Save processed CSV + metrics + experiment log

In [ ]:
save_processed_dataset(df_processed, PROCESSED_PATH)

# Also save a lean training-only CSV for convenience (optional artefact)
train_only_path = Path(cfg.paths.processed_data).with_name("cssrs_processed_train_view.csv")
save_processed_dataset(train_view, train_only_path)

dist_after.to_csv(METRICS_DIR / "preprocessing_class_distribution.csv", index=False)
compare.to_csv(METRICS_DIR / "preprocessing_class_count_delta.csv", index=False)

timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
metrics_payload = {
    "timestamp_utc": timestamp,
    "stage": "preprocessing",
    "notebook": "02_Preprocessing.ipynb",
    "dataset_name": cfg.experiment.dataset_name,
    "raw_path": str(cfg.paths.raw_data),
    "processed_path": str(PROCESSED_PATH),
    "train_view_path": str(train_only_path),
    "seed": cfg.SEED,
    "report": report_dict,
    "config": {
        "unicode_form": PP.unicode_form,
        "unescape_html": bool(PP.unescape_html),
        "drop_exact_duplicates": bool(PP.drop_exact_duplicates),
        "drop_duplicate_text": bool(PP.drop_duplicate_text),
        "output_columns": output_columns,
    },
    "hyperparameters_snapshot": {
        "model_name": cfg.MODEL_NAME,
        "learning_rate": cfg.LEARNING_RATE,
        "batch_size": cfg.BATCH_SIZE,
        "epochs": cfg.EPOCHS,
        "dropout": cfg.DROPOUT,
        "seed": cfg.SEED,
        "optimizer": cfg.training.optimizer,
        "scheduler": cfg.training.scheduler,
    },
}

metrics_path = METRICS_DIR / "preprocessing_report.json"
with metrics_path.open("w", encoding="utf-8") as fh:
    json.dump(metrics_payload, fh, indent=2, ensure_ascii=False)

log_path = Path(cfg.paths.experiment_log)
log_path.parent.mkdir(parents=True, exist_ok=True)
with log_path.open("a", encoding="utf-8") as fh:
    fh.write(json.dumps({
        "timestamp_utc": timestamp,
        "stage": "preprocessing",
        "notebook": "02_Preprocessing.ipynb",
        "dataset_name": cfg.experiment.dataset_name,
        "n_input_rows": report.n_input_rows,
        "n_output_rows": report.n_output_rows,
        "seed": cfg.SEED,
        "model_name": cfg.MODEL_NAME,
        "learning_rate": cfg.LEARNING_RATE,
        "batch_size": cfg.BATCH_SIZE,
        "epochs": cfg.EPOCHS,
        "dropout": cfg.DROPOUT,
        "optimizer": cfg.training.optimizer,
        "scheduler": cfg.training.scheduler,
        "processed_path": str(PROCESSED_PATH),
    }, ensure_ascii=False) + "\n")

logger.info("Preprocessing report → %s", metrics_path)
print("Saved:")
print(f"  {PROCESSED_PATH}")
print(f"  {train_only_path}")
print(f"  {metrics_path}")
print(f"  {log_path}")

## 9. Summary for the paper / next step

| Artefact | Path |
|----------|------|
| Full processed CSV (incl. LLM labels for later benchmark) | `DATA/processed/cssrs_processed.csv` |
| Training view (`content`, `severity`, metadata) | `DATA/processed/cssrs_processed_train_view.csv` |
| Audit JSON | `RESULTS/metrics/preprocessing_report.json` |
| Class delta table | `RESULTS/metrics/preprocessing_class_count_delta.csv` |

### Implications for Notebook 3

- Tokenize **processed** `content` with `mental/mental-bert-base-uncased`.
- Set `tokenizer.truncation_side = "left"` (clinical climax / recent intent often appears late in posts).
- Recommend `max_length` from MentalBERT subword length percentiles — not the whitespace proxy from EDA.

---

**Stop here.** Await approval before generating Notebook 3 (Tokenization).